In [3]:
with open('input.txt', 'r', encoding='utf-8') as file:
    text = file.read()

In [4]:
print(f"Length of the text: {len(text)} characters")

Length of the text: 1115394 characters


In [6]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Vocabulary size: {vocab_size} characters")
print(f"Characters: {chars}")

Vocabulary size: 65 characters
Characters: ['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']



## Tokenizer

In [8]:
stoi = { ch: i for i, ch in enumerate(chars) }
itos = { i: ch for i, ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]  # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l])  # decoder

print(encode("!$ABC"))
print(decode(encode("!$ABC")))

[2, 3, 13, 14, 15]
!$ABC


In [10]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(f"Data tensor shape: {data.shape}, dtype: {data.dtype}")
print(data[:1000])  # print the first 1000 characters as integers

Data tensor shape: torch.Size([1115394]), dtype: torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,

## Train and Validation Split

In [11]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [12]:
block_size = 8  # context length for predictions
train_data[:block_size+1]  # show the first block_size+1 characters of the training data

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [13]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"Context (input): {context.tolist()} -> '{decode(context.tolist())}'")
    print(f"Target (output): {target.item()} -> '{decode([target.item()])}'")
    print()

Context (input): [18] -> 'F'
Target (output): 47 -> 'i'

Context (input): [18, 47] -> 'Fi'
Target (output): 56 -> 'r'

Context (input): [18, 47, 56] -> 'Fir'
Target (output): 57 -> 's'

Context (input): [18, 47, 56, 57] -> 'Firs'
Target (output): 58 -> 't'

Context (input): [18, 47, 56, 57, 58] -> 'First'
Target (output): 1 -> ' '

Context (input): [18, 47, 56, 57, 58, 1] -> 'First '
Target (output): 15 -> 'C'

Context (input): [18, 47, 56, 57, 58, 1, 15] -> 'First C'
Target (output): 47 -> 'i'

Context (input): [18, 47, 56, 57, 58, 1, 15, 47] -> 'First Ci'
Target (output): 58 -> 't'



## Data Loader Batches and Chunks

In [19]:
torch.manual_seed(1337)
batch_size = 4  # how many independent sequences will we process in parallel?
block_size = 8  # what is the maximum context length for predictions?

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb , yb = get_batch('train')
print("Inputs (x):")
print(xb.shape)
print(xb)
print("Targets (y):")
print(yb.shape)
print(yb)

print("-----")

for b in range(batch_size):  # batch dimension
    for t in range(block_size):  # time dimension
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"when input is {context.tolist()} -> '{decode(context.tolist())}', the target: {target.item()} -> '{decode([target.item()])}'")

Inputs (x):
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
Targets (y):
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
-----
when input is [24] -> 'L', the target: 43 -> 'e'
when input is [24, 43] -> 'Le', the target: 58 -> 't'
when input is [24, 43, 58] -> 'Let', the target: 5 -> '''
when input is [24, 43, 58, 5] -> 'Let'', the target: 57 -> 's'
when input is [24, 43, 58, 5, 57] -> 'Let's', the target: 1 -> ' '
when input is [24, 43, 58, 5, 57, 1] -> 'Let's ', the target: 46 -> 'h'
when input is [24, 43, 58, 5, 57, 1, 46] -> 'Let's h', the target: 43 -> 'e'
when input is [24, 43, 58, 5, 57, 1, 46, 43] -> 'Let's he', the target: 39 -> 'a'
when input is [44] -> 'f', the target: 53 -> 'o'
when input is [44, 53]

## BigramModel

In [24]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx)  # (B,T,C)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B,T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :]  # becomes (B,C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1)  # (B,C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)  # (B,1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1)  # (B,T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print("Logits shape:", logits.shape)
print("Loss:", loss.item())

# Generate some text
idx = torch.zeros((1, 1), dtype=torch.long)
generated = decode(m.generate(idx, max_new_tokens=100)[0].tolist())
print("Generated text:", generated)

Logits shape: torch.Size([32, 65])
Loss: 4.878634929656982
Generated text: 
SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


## Train our Model

In [34]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [37]:
batch_size = 32
for steps in range(100000):
    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(f"Step {steps}: loss {loss.item()}")

Step 99999: loss 2.429525136947632


In [35]:
print(decode(m.generate(torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))



Hersl!
CAns and d m owbe thengse at'ge d:' cus evefllmerancour he IOLARome atthal s,
asade d?
INERAMaifofollocrarecepourine heanef th rof nghe I lle. n f wn y waleereand bustad Wh th, t.
Anthay t maghe tan ise, mate ickin
Be.
Y ngry th g chien:
Thed, t t,
CABUSime;
Thent.
'le r ENI d mat IED: f talage, thiotroche avoou winds anglfounde elve wimed Anirne ch e PO: pode, ss,
te'sthe t ty, iserutor:
She t chaventhand gend s; CI co; inut'se!
Jun anounde g IVIfotto me h enkedoudin. se s, fofote bicar
